## Zaawansowane przetwarzanie języka naturalnego
# Moduł 5: Duże modele językowe

Ćwiczenie będzie wykorzystywało system *Ollama*, aby uruchomić lokalnie serwer hostujący duży model językowy *(Large Language Model, LLM)*. System wystarczy ściągnąć i rozpakować, tj. można go zainstalować nawet bez uprawnień administratora. Dodatkowo Ollama oferuje dostęp do szerokiej gamy modeli open-weights, które są skwantyzowane ("skompresowane"). 

W ćwiczeniu będziemy używać małego modelu językowego, aby móc je przeprowadzić na CPU (choć będzie to wolne). Potrzebne jednak będzie ok. 4 GB wolnego miejsca na dysku. Jeśli masz dostęp do CUDA, Ollama automatycznie go użyje. Dla kart graficznych AMD można zainstalować specjalną kompilację biblioteki dla tych kart.

*Uwaga:* Biblioteka Ollama została wybrana ze względu na prostotę instalacji. Jest ona zupełnie wystarczające do osobistego wykorzystywania LLMów, jednak _nie_ jest zalecana do profesjonalnych instalacji lub przetwarzania dużej ilości zapytań. Patrz: biblioteka `vllm`.

*Uwaga 2:* Modele "open-weights" są pewną analogią do oprogramowania open source. Wytrenowane wagi modeli open-weight można ściągnąć bezpłatnie z internetu i wykorzystywać zgodnie z licencją, jednak dane czy hiperparametry użyte do ich treningu nie są otwarte. Nie mamy również dostępu do kodu źródłowego który posłużył do ich trenowania.

## Zadanie 1


Na początku ściągnijmy i rozpakujmy bibliotekę Ollama. Podany link zawiera paczekę dla systemu Linux. Dla innych systemów operacyjnych: znajdź odpowiednią paczkę lub instalator na stronie [ollama.com](https://ollama.com)


In [1]:
!curl -fsSL https://ollama.com/download/ollama-linux-amd64.tgz | tar zx -C .

Zainstaluj także paczkę Pythonową umożliwiającą korzystanie z API Ollamy.

In [2]:
!pip install ollama

Defaulting to user installation because normal site-packages is not writeable


Otwórz w nowym okienku terminal i uruchom serwer Ollama dzialąjąc z folderu w którym rozpakowałeś instalację.

```
./bin/ollama serve
```
Ściągnij najmniejszy model LLM z rodziny `gemma3`. (Zadanie można wykorzystać z dowolnym innymi modelem)

In [3]:
!./bin/ollama pull gemma3:270m

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 735af2139dc6: 100% ▕██████████████████▏ 291 MB                         
pulling 4b19ac7dd2fb: 100% ▕██████████████████▏  476 B                         
pulling 3e2c24001f9e: 100% ▕██████████████████▏ 8.4 KB                         
pulling 339e884a40f6: 100% ▕██████████████████▏   61 B                         
pulling 74156d92caf6: 100% ▕██████████████████▏  490 B                         
verifying sha256 digest 
writing manifest 
success 


Spróbujmy wygenerować jakiś tekst LLMem. Większość LLMów jest trenowane do wykonywania instrukcji wydanych przez użytkownika w schemacie konwersacyjnym. Wejście do LLMa składa się więc z naprzemiennych tekstów wydawanych przez użytkownika i odpowiedzi generowanych przez agenta. 

Jeśli nasza konwersacja z modelem wygląda w ten sposób:

>- Ile kół ma auto?
>- Cztery
>- A autobus?

to wejście do modelu językowego będzie wyglądało następująco:
```
START_USER Ile kół ma auto? STOP_USER START_ASSISTANT Cztery STOP_ASSISTANT START_USER A autobus? STOP_USER START_ASSISTANT
```
a zadaniem modelu będzie dokończenie tak skonstruowanego "zdania". Zwróć uwagę, że zamiast używanych na wykładzie tokenów `START`/`STOP` w wejściu pojawiają się pary tych tokenów z określeniem roli wypowiadającego zdanie.

Szczegółowe nazwy tokenów początkowych/kończących i sposób formatowania wejścia są różne dla modeli różnych dostawców i wymagają sprawdzenia w dokumentacji modelu. Na szczęście, Ollama przeprowadza tego typu konwersję automatycznie, a my możemy podać wejście w formacie listy poszczególnych wypowiedzi.


In [4]:
llm_input = [
  {
    'role': 'user',
    'content': 'Ile kół ma auto?',
  },
  {
    'role': 'assistant',
    'content': 'Cztery.',
  },
  {
    'role': 'user',
    'content': 'A autobus?',
  }
]

Spróbujmy uzyskać odpowiedź asystenta dla tak zaczętej konwersacji. W tym celu wykorzystamy funkcję `chat` która jako parametr przyjmuje nazwę modelu LLM oraz listę wiadomości (wejście do modelu).

In [5]:
from ollama import chat
from ollama import ChatResponse

response: ChatResponse = chat(model='gemma3:270m', messages=llm_input)

print(response['message']['content'])

A autobus ma 4 osoby.


I gotowe! Uzyskana odpowiedź może nie być satysfakcjonująca, bo używany model jest bardzo mały. Jakość przetwarzania języków innych niż-angielski najczęściej szybciej spada w czasie "zmniejszania" modeli niż języka angielskiego. Nasz dialog przeprowadziliśmy w języku polskim, dla którego - tak czy tak - jest mniej danych uczących niż dla języka angielskiego.

Spójrz na strukturę zwracanej odpowiedzi - zwróć uwagę że zwrócono wiadomość z rolą `assistant`. Gdybyś chciał kontynuować konwersację z modelem, musiałbyś dokleić ją do listy `llm_input` i dodatkowo dodać do niej także wypowiedź użytkownika.

In [6]:
response

ChatResponse(model='gemma3:270m', created_at='2026-01-10T22:27:33.686898196Z', done=True, done_reason='stop', total_duration=3440409883, load_duration=2404851323, prompt_eval_count=31, prompt_eval_duration=218735526, eval_count=8, eval_duration=436090738, message=Message(role='assistant', content='A autobus ma 4 osoby.', thinking=None, images=None, tool_name=None, tool_calls=None), logprobs=None)

Twoja kolej: Uzyskaj od modelu odpowiedź na pytanie "How to pass Advanced NLP course at Poznan University of Technology?". 

In [7]:

llm_input = [
    {
        'role': 'user',
        'content': 'How to pass Advanced NLP course at Poznan University of Technology?',
    }
]

response: ChatResponse = chat(model='gemma3:270m', messages=llm_input)

print(response['message']['content'])

Passing Advanced NLP course at Poznan University of Technology requires a combination of strong foundational knowledge, practical skills, and a comprehensive understanding of the subject. Here's a breakdown of how to approach it:

**1. Strong Foundation in NLP:**

*   **Core Concepts:**
    *   **Natural Language Understanding (NLU):**  This is the foundational skill. Understanding the meaning of text, identifying entities, relationships, and context.
    *   **Machine Learning (ML):**  Understanding how to train models to predict and classify text.
    *   **Natural Language Generation (NLG):**  The ability to translate between human language and text, and to generate coherent and fluent text.
    *   **Text Preprocessing:**  Handling noise, formatting, and special characters in text.
    *   **Advanced NLP Techniques:**  Leveraging cutting-edge techniques like transformers, recurrent neural networks (RNNs), and generative models.
*   **Knowledge Gaps:**
    *   **Generalization:**  T

Następnie, _kontynuując_ konwersację z modelem, dopytaj się model czy wie ile osób zdaje kurs w pierwszym terminie.

In [8]:
llm_input.append({
    'role': 'user',
    'content': 'Do you know how many students pass the course in the first attempt? Tell me how many.',
})

response: ChatResponse = chat(model='gemma3:270m', messages=llm_input)

print(response['message']['content'])

Yes, I do. The Advanced NLP course at Poznan University of Technology is a highly regarded and well-regarded program.



## Zadanie 2

Wykorzystaj model LLM do zbudowania klasyfikatora wydźwięku. Do testów modelu możesz wykorzystać poniższy mały zbiór danych.

In [9]:
examples = [
    ("I absolutely loved this movie!", "positive"),
    ("This is the worst service I have ever experienced.", "negative"),
    ("The product works fine, nothing special.", "neutral"),
    ("I am so happy with my purchase.", "positive"),
    ("I regret buying this, it broke after one day.", "negative"),
    ("The meeting was okay, not great but not terrible.", "neutral"),
    ("Fantastic experience, I will definitely come back!", "positive"),
    ("I’m really disappointed with the quality.", "negative"),
    ("It does what it’s supposed to do.", "neutral"),
    ("Amazing support team, very helpful!", "positive"),
]

Napisz odpowiednią instrukcję dla modelu, w której zdefiniuj zadanie analizy wydźwięku i sprawdź odpowiedzi modelu na `examples`. 

In [10]:
instruction = (
    "You are a sentiment analysis assistant. "
    "Classify each input sentence into one of three classes: 'positive', 'negative', or 'neutral'. "
    "Only output the class label."
)

for text, true_label in examples:
    llm_input = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": text},
    ]
    
    response: ChatResponse = chat(model='gemma3:270m', messages=llm_input)
    
    predicted_label = response['message']['content'].strip()
    print(f"Text: {text}")
    print(f"True label: {true_label}, Predicted label: {predicted_label}")
    print("-" * 60)

Text: I absolutely loved this movie!
True label: positive, Predicted label: Positive
------------------------------------------------------------
Text: This is the worst service I have ever experienced.
True label: negative, Predicted label: Negative
------------------------------------------------------------
Text: The product works fine, nothing special.
True label: neutral, Predicted label: Positive
------------------------------------------------------------
Text: I am so happy with my purchase.
True label: positive, Predicted label: positive
------------------------------------------------------------
Text: I regret buying this, it broke after one day.
True label: negative, Predicted label: negative
------------------------------------------------------------
Text: The meeting was okay, not great but not terrible.
True label: neutral, Predicted label: positive
------------------------------------------------------------
Text: Fantastic experience, I will definitely come back!
True

Rozszerz swoją instrukcję i podaj modelowi jeden przykład uczący, pokazujący mu jak należy wykonać zadanie (*one-shot learning*). Sprawdź działanie modelu.

In [11]:
instruction = (
    "You are a sentiment analysis assistant. "
    "Classify each input sentence into one of three classes: 'positive', 'negative', or 'neutral'. "
    "Only output the class label."
)

one_shot_example = {
    "role": "user",
    "content": "The exaple sentence: 'I like this phone!' should be classified as 'positive'"
}

for text, true_label in examples:
    llm_input = [
        {"role": "system", "content": instruction},
        one_shot_example,
        {"role": "user", "content": f" Sentence to classify is '{text}'"}
    ]
    
    response: ChatResponse = chat(model='gemma3:270m', messages=llm_input)
    
    predicted_label = response['message']['content'].strip()
    print(f"Text: {text}")
    print(f"True label: {true_label}, Predicted label: {predicted_label}")
    print("-" * 60)


Text: I absolutely loved this movie!
True label: positive, Predicted label: positive
------------------------------------------------------------
Text: This is the worst service I have ever experienced.
True label: negative, Predicted label: Positive
------------------------------------------------------------
Text: The product works fine, nothing special.
True label: neutral, Predicted label: negative
------------------------------------------------------------
Text: I am so happy with my purchase.
True label: positive, Predicted label: Positive
------------------------------------------------------------
Text: I regret buying this, it broke after one day.
True label: negative, Predicted label: Positive
------------------------------------------------------------
Text: The meeting was okay, not great but not terrible.
True label: neutral, Predicted label: Positive
------------------------------------------------------------
Text: Fantastic experience, I will definitely come back!
True

W praktycznych aplikacjach - takich jak budowa klasyfikatora - często chcemy uzyskać od modelu odpowiedź w określonym formacie, aby umożliwić dalsze przetwarzanie odpowiedzi z modelu. Popularnym sposobem definiowania oczekiwanych danych jest określenie formatu struktury JSON. W jaki jednak sposób zagwarantować, że dostaniemy od LLMa oczekiwany plik JSON?

Tu przychodzi nam z pomocą funkcjonalność nazywana *Structured Outputs*, która umożliwia wymuszenie odpowiedzi o określonym formacie... choć może się to nie udać! Dlaczego? Funkcjonalność ta nie w sobie żadnej magii, a po prostu modyfikuje ona algorytm dekodujący tekst z LLM. Algorytm analizuje przekazany schemat specyfikujący format oczekiwanego JSONa i maskuje (zeruje prawdopodobieństwa) tokenów, których wygenerowanie spowodowałoby złamanie specyfikacji formatu. Sam LLM oblicza więc prawdopodobieństwa następnych tokenów w standardowy sposób, ale z obliczonego przez niego wyniku "na siłę" zerowane są prawdopodobieństwa niepożądanych tokenów i dopiero wtedy wybiera się/losuje się następny wygenerowany token. Istnieje więc możliwość, że np. LLM nigdy (a przynajmniej w przewidzianej maksymalnej liczbie tokenów) nie wygeneruje kończącego JSONa znaku `}`, a zwrócona odpowiedź nie będzie miała oczekiwanego formatu. 

Z tego powodu oprócz podania schematu JSON funkcjonalności Structured Outputs, modyfikującej dekodowanie odpowiedzi, dobrze jest dodatkowo poinformować model LLM w tekście instrukcji, jakiego typu odpowiedzi się spodziewamy. Sama informacja "Format the answer as JSON" już może uprościć zadanie modelowi, choć zwykle lepiej jest opisać schemat dokładnie.


Przeanalizujmy przykładowy kod definiujący oczekiwany schemat JSONa za pomocą biblioteki `pydantic`.


In [12]:
from pydantic import BaseModel

# Definiujemy obiekt, który określa oczekiwane pola w obiekcie JSON 

class Country(BaseModel):
  name: str
  capital: str
  languages: list[str]

json_schema = Country.model_json_schema()

print(json_schema)

{'properties': {'name': {'title': 'Name', 'type': 'string'}, 'capital': {'title': 'Capital', 'type': 'string'}, 'languages': {'items': {'type': 'string'}, 'title': 'Languages', 'type': 'array'}}, 'required': ['name', 'capital', 'languages'], 'title': 'Country', 'type': 'object'}


Wymagamy więc obiektu JSON o 3 możliwych `properties`, a obecność każdej z nich jest wymagana (lista `required` specyfikacji). Ponadto określone są typy pół/właściwości JSONa

Taki schemat JSONa można także np. napisać ręcznie lub uzyskać w inny sposób. Aby uzyskać od modelu odpowiedź w oczekiwanym formacie należy podać schemat JSONa do argumentu `format`.

In [13]:
response = chat(
  model='gemma3:270m',
  messages=[{'role': 'user', 'content': 'Tell me about Poland.'}],
  format= json_schema,
)

print(response['message']['content'])

{
"name": "Poland",
"capital": "Warsaw",
"languages": ["Polish", "German", "French", "Russian", "Italian", "Basque", "Andorian", "Welsh", "Czech", "Slovak", "Slovenian", "Croatian", "Bulgarian", "Serbian", "Ukrainian", "Moldovan", "Latvian", "Lithuanian", "Turcan", "Slovakian", "Polish", "German", "French", "Russian", "Italian", "Basque", "Andorian", "Welsh", "Czech", "Slovakian", "Slovenian", "Croatian", "Bulgarian", "Serbian", "Ukrainian", "Moldovan", "Latvian", "Lithuanian", "Turcan", "Slovakian", "Polish", "German", "French", "Russian", "Italian", "Basque", "Andorian", "Welsh", "Czech", "Slovakian", "Slovenian", "Croatian", "Bulgarian", "Serbian", "Ukrainian", "Moldovan", "Latvian", "Lithuanian", "Turcan", "Slovakian", "Polish", "German", "French", "Russian", "Italian", "Basque", "Andorian", "Welsh", "Czech", "Slovakian", "Slovenian", "Croatian", "Bulgarian", "Serbian", "Ukrainian", "Moldovan", "Latvian", "Lithuanian", "Turcan", "Slovakian", "Polish", "German", "French", "Russian",

Dodajmy do instrukcji informację o spodziewanym formacie odpowiedzi.

In [14]:
response = chat(
  model='gemma3:270m',
  messages=[{'role': 'user', 'content': 'Tell me about Poland. Mention: name, capital and official languages. Format the answer as JSON.'}],
  format= json_schema,
)

print(response['message']['content'])

{"name": "Poland", "capital": "Warsaw", "languages": ["Polish", "English"]}


Zaletą korzystania z biblioteki `pydantic` jest to, że możemy skonwertować uzyskany JSON na obiekt pythonowy o wcześniejszej specyfikacji.



In [15]:
country = Country.model_validate_json(response.message.content)
print(country)

name='Poland' capital='Warsaw' languages=['Polish', 'English']


Zwróć uwagę że funkcja `model_validate_json` może także zwrócić wyjątek, gdyby wygenerowana odpowiedź nie była zgoda ze schematem obiektu.


Popraw implementację swojego klasyfikatora bazującego na LLM, tak aby zwracał tylko jedną z trzech odpowiedzi: "positive", "negative" lub "neutral". Wykorzystaj do tego obiekt typu `Enum` i zdefiniuj JSON `{"class": "..."}`.

In [16]:
from enum import Enum
from pydantic import BaseModel
from ollama import chat, ChatResponse

class Sentiment(str, Enum):
    positive = "positive"
    negative = "negative"
    neutral = "neutral"

class SentimentOutput(BaseModel):
    class_: Sentiment

instruction = (
    "You are a sentiment analysis assistant. "
    "Classify each input sentence into one of three classes: 'positive', 'negative', or 'neutral'. "
    "Format the answer as JSON like this: {\"class\": \"<class>\"}. "
    "Only output the JSON object and nothing else."
)

one_shot_example = {
    "role": "user",
    "content": "The example sentence: 'I love this phone!' should be classified as positive. "
               "Format as JSON: {\"class\": \"positive\"}."
}

for text, true_label in examples:
    llm_input = [
        {"role": "system", "content": instruction},
        one_shot_example,
        {"role": "user", "content": f"Sentence to classify: '{text}'"}
    ]
    
    response: ChatResponse = chat(
        model='gemma3:270m',
        messages=llm_input,
        format=SentimentOutput.model_json_schema()
    )
    
    try:
        sentiment_obj = SentimentOutput.model_validate_json(response['message']['content'])
        predicted_label = sentiment_obj.class_
    except Exception as e:
        predicted_label = f"Error: {e}"
    
    print(f"Text: {text}")
    print(f"True label: {true_label}")
    print(f"Predicted JSON: {response['message']['content']}")
    print(f"Predicted label: {predicted_label}")
    print("-" * 60)

Text: I absolutely loved this movie!
True label: positive
Predicted JSON: {"class_": "positive"}
Predicted label: positive
------------------------------------------------------------
Text: This is the worst service I have ever experienced.
True label: negative
Predicted JSON: {"class_": "negative"}

Predicted label: negative
------------------------------------------------------------
Text: The product works fine, nothing special.
True label: neutral
Predicted JSON: {"class_": "neutral"}
Predicted label: neutral
------------------------------------------------------------
Text: I am so happy with my purchase.
True label: positive
Predicted JSON: {"class_": "positive"}
Predicted label: positive
------------------------------------------------------------
Text: I regret buying this, it broke after one day.
True label: negative
Predicted JSON: {"class_": "negative"}
Predicted label: negative
------------------------------------------------------------
Text: The meeting was okay, not grea

*Ćwiczenie*
1. Oprócz ról: `user` i `assistant`, większość LLMów posiada także specjalną rolę `system`. Sprawdź w Internecie do czego ona służy. Zastanów się, do jakich zastosowań jest ona przydatna (np. skąd LLM może wiedzieć, jaki mamy dzisiaj dzień?)

2. LLMy pomimo wielu zdolności często mają problem w odpowiedzeniu na pytania dotyczące budowy wyrazów np. "Ile razy litera "w" występuje w słowie Warszawa?" Dlaczego? 

**Odpowiedzi:**

1. Rola `system` pozwala dostarczyć modelowi zewnętrzną, „stałą” instrukcję lub kontekst, który wpływa na całe zachowanie modelu w konwersacji. Może określać ton, styl odpowiedzi, zasady działania lub przekazywać informacje, których model sam z danych treningowych nie zna, np. aktualną datę.

2. LLMy operują na tokenach, nie na pojedynczych literach, więc nie widzą tekstu na poziomie liter. Zadanie liczenia wystąpień litery wymaga operacji na poziomie symboli, a nie tokenów, co sprawia, że model często nie jest w stanie udzielić poprawnej odpowiedzi.
